# CSM (lxing532/Dialogue-Topic-Segmenter) — 준-완벽 재현 (Colab)

**실행 순서 (중요 — 커널 재시작 1회 포함)**
1. (선택) `[0]` GPU 확인
2. **`[1]`** 클론+의존성 설치 → **셀 끝에서 커널 자동 재시작됨(정상)**. 재연결 대기.
3. 재시작 후 **`[1b]`** 부터 → `[2]`(원본 DailyDialog) → 필요시 `[2b]` 업로드 후 `[2]` 재실행 → `[3]` 학습 → `[4]`/`[4b]` 평가 → `[5]` zip 다운로드. **`[1]` 은 다시 실행 금지.**
   - 세션이 한 번 꼬였으면 **Runtime ▸ Disconnect and delete runtime** 로 완전 새 세션에서 1번부터.

**재현 등급 = 준-완벽 (사용자 결정: 옵티마이저까지 핀)**
- `transformers==4.39.3` — Colab py3.12 에서 논문핀 4.27.4 는 설치 불가(tokenizers 휠 없음). 4.39.3 도 `transformers.AdamW`(<4.40) 존재 → **train.py 무패치, 옵티마이저 동일**.
- HP 정확: epochs=10, batch=24, margin=1, encoder=`aws-ai/dse-bert-base`, `AdamW(lr=2e-5, eps=1e-8)`, val 10%.
- 데이터 정확: 원본 `ijcnlp_dailydialog.zip` 의 `dialogues_{text,topic,act}.txt` **verbatim** (`[2b]` 로 업로드). roskoN/HF 는 topic 없음 → 합성 금지.
- metric 핀: `segeval==2.0.11`. 완화: `torch`(Colab 기본), `transformers` 4.39.3(≠4.27.4) → REPORT 에 "준-완벽" 명시.
- 한계: 레포 seed 미고정 → run 간 비결정성(bitwise 동일 아님).

> 임계경로 = 원본 `ijcnlp_dailydialog.zip`(topic 포함) 확보 → `[2b]` 업로드. superseg 는 ckpt 회수 후 로컬 Hi-EM harness 별도. Runtime ▸ GPU(T4+).

In [ ]:
# [0] GPU / 환경 확인
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! GPU 없음 — Runtime>Change runtime type>GPU 로 바꾸고 재실행 권장 (CPU 학습은 매우 느림)')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# [1] 클론 + 의존성 핀 설치 → **커널 자동 재시작**
#   ※ 이 셀은 끝에서 커널을 강제 재시작합니다(정상). 재시작 후 [1b] 부터 이어 실행.
#   ※ transformers 4.27.4 는 Colab py3.12 설치 불가(tokenizers 휠 없음) → 4.39.3
#     (transformers.AdamW 존재 <4.40 → train.py 무패치, 준-완벽재현).
import os
REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'
os.makedirs('/content/csm', exist_ok=True)
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/lxing532/Dialogue-Topic-Segmenter.git {REPO_DIR}
# 재현 핵심 핀만 (한 방에). sentence-transformers 는 우리 eval(NSP/CM)에 불필요 → 제외(충돌 회피)
!pip -q install 'transformers==4.39.3' 'numpy<2' 'segeval==2.0.11' 'scikit-learn>=1.2' 'nltk==3.8.1' 'huggingface_hub>=0.20,<0.26' 2>&1 | tail -3
print('\\n==== 설치 완료. 커널을 재시작합니다 (정상) ====')
print('재시작 후: [1b] 셀부터 순서대로 실행하세요. [1] 은 다시 실행하지 마세요.')
import os as _os
_os.kill(_os.getpid(), 9)   # 강제 커널 재시작 (pip 반영 위해 필수)

In [ ]:
# [1b] (커널 재시작 *후* 실행) — 검증 + 경로 재설정
#   [1] 이 커널을 죽였으므로 변수 초기화됨. 여기서 다시 세팅. 이후 [2]→[5] 순서대로.
import os
REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'
assert os.path.isdir(REPO_DIR), '[1] 을 먼저 실행(클론)하세요'
os.chdir(REPO_DIR)
import torch, transformers, nltk
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda', torch.cuda.is_available())
assert transformers.__version__ == '4.39.3', \
    f'transformers 4.39.3 필요(got {transformers.__version__}) — [1] 재실행/런타임 새로'
from transformers import AdamW  # <4.40 → train.py 무패치 재현 (준-완벽: transformers 4.39.3)
print('transformers.AdamW OK → train.py 무패치. 재현등급=준(transformers 4.39.3)')
for pkg in ['stopwords', 'punkt']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print('nltk warn', pkg, e)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo files:', sorted(os.listdir(REPO_DIR)))
print('eval datasets:', sorted(os.listdir(os.path.join(REPO_DIR, 'data', 'eval'))))

In [ ]:
# [2] DailyDialog **원본** 준비 (완벽재현 = 원본 ijcnlp_dailydialog 4파일 verbatim)
#   roskoN/HF 는 topic 없음 → 재현 불가. 반드시 원본 ijcnlp_dailydialog.zip.
#   원본 zip 안: ijcnlp_dailydialog/{dialogues_text,topic,act,emotion}.txt (full).
#   ※ yanran.li 죽음 → 자동 다운로드 없음. 사람이 [2b] 로 업로드해야 함.
import os, glob, zipfile, shutil
DD_DIR = os.path.join(REPO_DIR, 'data', 'train', 'dailydialog')
os.makedirs(DD_DIR, exist_ok=True)
NEED = ['dialogues_text.txt', 'dialogues_topic.txt', 'dialogues_act.txt']

def have_dd():
    return all(os.path.isfile(os.path.join(DD_DIR, f)) and
               os.path.getsize(os.path.join(DD_DIR, f)) > 0 for f in NEED)

if not have_dd():
    # 업로드된 zip 후보 탐색 → 유효한 zip 만. 깨진/HTML 가짜 .zip 은 삭제(독 제거)
    raw = (glob.glob('/content/**/ijcnlp_dailydialog*.zip', recursive=True) +
           glob.glob('/content/**/*dailydialog*.zip', recursive=True) +
           glob.glob('/content/*.zip'))
    cand = []
    for p in dict.fromkeys(raw):
        if zipfile.is_zipfile(p):
            cand.append(p)
        else:
            print('무효 zip(삭제):', p, '(HTML/손상 — 진짜 원본 아님)')
            try: os.remove(p)
            except OSError: pass
    if cand:
        with zipfile.ZipFile(cand[0]) as z:
            z.extractall('/content/dd_extract')
        for zp in glob.glob('/content/dd_extract/**/*.zip', recursive=True):
            if zipfile.is_zipfile(zp):
                with zipfile.ZipFile(zp) as z:
                    z.extractall(os.path.dirname(zp))
        for name in NEED + ['dialogues_emotion.txt']:
            hits = sorted(glob.glob(f'/content/dd_extract/**/{name}', recursive=True),
                          key=lambda p: ('train' in p or 'valid' in p or 'test' in p, len(p)))
            if hits:
                shutil.copy(hits[0], os.path.join(DD_DIR, name))
                print('placed', name, '<-', hits[0])
    else:
        print('!! 유효한 원본 ijcnlp_dailydialog.zip 없음 →'
              ' [2b] 셀로 진짜 원본 zip 업로드 후 이 [2] 셀 재실행')

if have_dd():
    shutil.copy(os.path.join(DD_DIR, 'dialogues_act.txt'),
                os.path.join(DD_DIR, 'dialogue_act.txt'))
    nl = {f: sum(1 for _ in open(os.path.join(DD_DIR, f))) for f in NEED}
    with open(os.path.join(DD_DIR, 'dialogues_topic.txt')) as fh:
        tline = fh.readline().strip()
    print('line counts:', nl, '| aligned:', len(set(nl.values())) == 1,
          '| topic int OK:', tline.isdigit(), '(sample=%r)' % tline)
    assert len(set(nl.values())) == 1 and tline.isdigit(), '원본 정렬/topic 형식 이상'
print('DailyDialog ready:', have_dd(), '| files:', sorted(os.listdir(DD_DIR)))

In [ ]:
# [2b] 원본 DailyDialog 업로드 (zip 또는 dialogues_*.txt 직접) → 그 다음 [2] 재실행
#   업로드 대상: dailydialog_original.zip  (또는 dialogues_text/topic/act.txt 3개)
from google.colab import files
import os, shutil
up = files.upload()
DD_DIR = '/content/csm/Dialogue-Topic-Segmenter/data/train/dailydialog'
os.makedirs(DD_DIR, exist_ok=True)
for fn in up:
    b = os.path.basename(fn)
    if b.endswith('.zip'):
        shutil.move(fn, os.path.join('/content', b))          # [2] 가 추출
        print('zip → /content/%s (이제 [2] 재실행)' % b)
    elif b.startswith('dialogues_') and b.endswith('.txt'):
        shutil.move(fn, os.path.join(DD_DIR, b))               # 바로 배치
        print('placed', b, '→', DD_DIR)
    else:
        shutil.move(fn, os.path.join('/content', b))
        print('saved /content/%s' % b)
print('다음: [2] 셀 실행해서 "DailyDialog ready: True" 확인')

In [ ]:
# [2c] 레포 버그 수정 + forward 성능 패치 (학습 수식/데이터/HP/optimizer 불변)
#   (1) BUG FIX: segment.py 가 없는 coherence_model 을 import(모듈 최상단)
#       → NSP·CM eval 전부 ModuleNotFoundError. 클래스는 model_utils 에 존재
#       → coherence_model.py shim 생성. ckpt state_dict 키 동일 → strict load OK.
#       (neural_texttiling CM: text_encoder([[tok]*3]) → patched forward 와
#        shape/역할순서/loss소비 동일. eval/분포상 동등 성능최적화.
#   (2) PERF: CoherenceNet.forward 가 batch 를 파이썬 루프로 BERT 샘플×3 직렬
#       호출 → step당 72 forward → [3B,128] 1회. train dropout RNG
#       흐름 상이로 bitwise 아님 = 분포상 동등 성능최적화(codex).
#   ※ train.py 의 DataLoader/워커/resume 은 [2e] 가 train.py 전체를 소유하며
#     처리(여기서 train.py 안 건드림 — 중복/충돌 방지). 모두 멱등.
import re, os
REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'

# (1) coherence_model.py shim — segment.py 의 missing import 해결
cm = os.path.join(REPO_DIR, 'coherence_model.py')
if os.path.exists(cm):
    print('coherence_model.py already exists (skip)')
else:
    open(cm, 'w').write('from model_utils import CoherenceNet\n')
    print('coherence_model.py shim 생성 (segment.py NSP/CM import 수정)')

# (2) batched CoherenceNet.forward (원본 per-sample 루프와 수치 동등)
mu = os.path.join(REPO_DIR, 'model_utils.py')
s = open(mu).read()
if '[perf] batched-equivalent' in s:
    print('model_utils.py already perf-patched (skip)')
else:
    NEW = '''    def forward(self, batch):
        # [perf] batched-equivalent of the original per-sample 3x BERT loop.
        B = len(batch)
        keys = list(batch[0][0].keys())
        big = {k: torch.cat([torch.cat([batch[i][r][k] for i in range(B)], 0)
                             for r in range(3)], 0).to(self.device)
               for k in keys}
        h = self.bert(**big).last_hidden_state[:, 0, :]      # [3B,768]
        dec = self.coherence_decoder(h)                      # [3B,2]
        sm = F.softmax(dec, dim=-1)                          # [3B,2]
        return sm.view(3, B, 2).permute(1, 0, 2).contiguous()  # [B,3,2]
'''
    s2 = re.sub(r"    def forward\(self, batch\):.*?return torch\.stack\(output, dim=0\)\n",
                NEW, s, count=1, flags=re.S)
    assert s2 != s and 'def forward' in s2, 'forward 패치 실패 — 원본 구조 확인'
    open(mu, 'w').write(s2)
    print('model_utils.py: batched forward (분포 동등 성능최적화)')
print('done — 알고리즘/데이터/HP/optimizer 불변. (DataLoader·resume 은 [2e])')

In [ ]:
# [2d] 체크포인트 Drive 영속화 — **이전 데이터와 격리된 전용 폴더**
#   ⚠ 기존 /content/drive/MyDrive/csm_ckpts (옛 warm-restart 25k) 및 그 외
#     Drive 파일은 절대 안 건드림/삭제 안 함. 새 깨끗한 단일-스케줄 run 은
#     아래 **전용 새 폴더**에만 씀 → 혼동·오염 없음.
#   train.py 는 이 폴더의 resume.pt 만 보고 이어감(없으면 fresh=논문 등가
#   단일 연속 학습 시작). cpt_<gstep>.pth 도 여기 저장(eval[4] 호환).
from google.colab import drive
import os
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/csm_steprresume_v2'   # 전용·신규
os.makedirs(CKPT_DIR, exist_ok=True)
_old = '/content/drive/MyDrive/csm_ckpts'
if os.path.isdir(_old):
    print('※ 옛 폴더', _old, '존재 — 사용/삭제 안 함(직접 정리하셔도 됨).')
_has = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pth') or f=='resume.pt']
print('CKPT_DIR =', CKPT_DIR,
      '| 기존 ckpt:', sorted(_has) if _has else '없음(→ fresh, 논문 등가 시작)')
print('이전 Drive 데이터와 격리됨. [3] 이 이 경로만 사용.')

In [ ]:
# [2e] TRUE STEP-RESUME 패치 (A, step 단위) — train.py 전체 재작성 + 로그폭주 방지
#   - resume.pt: model+optimizer+scheduler+gstep+epoch+step_in_epoch+total
#   - epoch 마다 seed 로 결정적 셔플 → 멈췄던 'epoch 의 그 step'으로 정확히
#     점프(재계산 0). LR 스케줄 전 구간 단일·연속, 모멘텀 유지 = 논문 등가.
#   - cpt_<gstep>.pth = 순수 model state_dict (eval[4] 호환) 계속 저장.
#   - DataLoader num_workers 포함([2c] train.py 패치 대체).
#   - tqdm mininterval=10s + data_utils drop-print 침묵 → Colab 폭주/멈춤 방지.
#   - seed42 = split+epoch-order+data_utils pseudo 결정화(codex 반영:
#     resume 세션간 pseudo셋 동일 → 진짜 단일연속). HP/loss 불변. 멱등.
import os, ast
REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'

du = os.path.join(REPO_DIR, 'data_utils.py')
s = open(du).read()
if "[silenced]" not in s:
    s = s.replace(
        "            print('[Error] Problematic datapoint/dialogue, dropped it...')\n",
        "            globals()['_NDROP']=globals().get('_NDROP',0)+1  # [silenced]\n")
    open(du, 'w').write(s); print('data_utils: drop-print 침묵화')
else:
    print('data_utils already silenced (skip)')

TRAIN_PY = r'''import argparse, os, torch
from torch.utils.data import DataLoader, Subset
from transformers import AdamW, get_linear_schedule_with_warmup
from data_utils import UtteranceDataset
from model_utils import CoherenceNet
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
SEED = 42

def parse_args():
    p=argparse.ArgumentParser()
    p.add_argument("-t","--dataset",default="./data/train/dailydialog")
    p.add_argument("-r","--epochs",type=int,default=10)
    p.add_argument("-b","--batch_size",type=int,default=24)
    p.add_argument("-m","--margin",type=float,default=1)
    p.add_argument("-e","--text_encoder",default="aws-ai/dse-bert-base")
    p.add_argument("-s","--checkpoints_path",default="./checkpoints/")
    return p.parse_args()

def collate_fn(b): return b

def marginal_ranking_loss(batch, margin):
    bt=batch[:, :, 0]
    l1=torch.nn.functional.relu(margin-(bt[:,0]-bt[:,1]))
    l2=torch.nn.functional.relu(margin-(bt[:,0]-bt[:,2]))
    l3=torch.nn.functional.relu(margin-(bt[:,1]-bt[:,2]))
    return torch.mean((l1+l2+l3)/3.0)

def validation_metric(sl):
    c=0
    for a,b,cc in sl:
        c+=sum(1 for x,y in [(a,b),(a,cc),(b,cc)] if x>y)
    return c/float(len(sl)*3) if sl else 0.0

def validation(model,vdl,device):
    cs=[]; model.eval()
    with torch.no_grad():
        for vb in vdl:
            o=model(vb); cs+=o[:,:,0].tolist()
    return validation_metric(cs)

def _save(model,opt,sch,gstep,epoch,sie,total,cps):
    torch.save(model.state_dict(), os.path.join(cps,"cpt_"+str(gstep)+".pth"))
    tmp=os.path.join(cps,"resume.pt.tmp")
    torch.save({"model":model.state_dict(),"opt":opt.state_dict(),
                "sched":sch.state_dict(),"gstep":gstep,"epoch":epoch,
                "sie":sie,"total":total}, tmp)
    os.replace(tmp, os.path.join(cps,"resume.pt"))

def epoch_order(n, epoch):
    g=torch.Generator(); g.manual_seed(SEED*10007+epoch)
    return torch.randperm(n, generator=g).tolist()

def train(model,full_train,vdl,opt,epochs,margin,device,cps,bs,nw):
    rp=os.path.join(cps,"resume.pt")
    bpe=(len(full_train)+bs-1)//bs            # batches per epoch
    if os.path.exists(rp):
        total=torch.load(rp,map_location="cpu")["total"]
    else:
        total=bpe*epochs
    sch=get_linear_schedule_with_warmup(opt,0,total)
    start_epoch=0; gstep=0; start_sie=0
    if os.path.exists(rp):
        ck=torch.load(rp,map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sch.load_state_dict(ck["sched"])
        gstep=ck["gstep"]; start_epoch=ck["epoch"]; start_sie=ck["sie"]
        print("[resume] epoch %d/%d step_in_epoch %d gstep %d (target %d) "
              "— 그 epoch 의 그 step 으로 점프"
              %(start_epoch+1,epochs,start_sie,gstep,total))
    else:
        print("[fresh] epoch1 step0 (target %d, 단일 LR 스케줄, seed %d)"
              %(total,SEED))
    lf=open("training_log.txt","a")
    for ei in range(start_epoch,epochs):
        order=epoch_order(len(full_train),ei)
        sie0=start_sie if ei==start_epoch else 0
        sub=Subset(full_train, order[sie0*bs:])
        dl=DataLoader(sub,batch_size=bs,shuffle=False,collate_fn=collate_fn,
                      num_workers=nw,pin_memory=True)
        print("\n======== Epoch %d / %d (start step %d/%d) ========"
              %(ei+1,epochs,sie0,bpe))
        lf.write("\n==== Epoch %d/%d start_sie %d ====\n"%(ei+1,epochs,sie0))
        lf.flush(); tl=0.0; model.train()
        for li,batch in tqdm(enumerate(dl),total=len(dl),desc="Training",
                             mininterval=10.0,dynamic_ncols=True):
            sie=sie0+li
            model.zero_grad()
            out=model(batch); loss=marginal_ranking_loss(out,margin)
            tl+=loss.item(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sch.step(); gstep+=1
            if sie%1000==0 and sie!=0:
                vr=validation(model,vdl,device)
                msg="e%d sie%d gstep%d loss%.4f val%s"%(
                    ei+1,sie,gstep,tl/max(1,(sie-sie0)),vr)
                print(msg); lf.write(msg+"\n"); lf.flush()
                _save(model,opt,sch,gstep,ei,sie+1,total,cps)
                model.train()
        _save(model,opt,sch,gstep,ei+1,0,total,cps)   # epoch 끝 → 다음 epoch
        print("=== epoch %d done avg_loss %.4f gstep %d"
              %(ei+1,tl/max(1,len(dl)),gstep))
    lf.close()
    print("[DONE] %d epoch 완료 = 논문 등가. 최종 gstep=%d"%(epochs,gstep))

def main():
    a=parse_args()
    import random
    random.seed(SEED); torch.manual_seed(SEED)
    device="cuda" if torch.cuda.is_available() else "cpu"
    enc=AutoModel.from_pretrained(a.text_encoder).to(device)
    tok=AutoTokenizer.from_pretrained(a.text_encoder)
    full=UtteranceDataset(os.path.join(a.dataset,"dialogues_text.txt"),
                          os.path.join(a.dataset,"dialogues_topic.txt"),
                          os.path.join(a.dataset,"dialogues_act.txt"),tok)
    g=torch.Generator(); g.manual_seed(SEED)
    perm=torch.randperm(len(full),generator=g).tolist()
    vs=int(0.1*len(full))
    val_idx=perm[:vs]; tr_idx=perm[vs:]
    nw=min(8,(os.cpu_count() or 4))
    vdl=DataLoader(Subset(full,val_idx),batch_size=a.batch_size,
                   shuffle=False,collate_fn=collate_fn,num_workers=nw,
                   pin_memory=True)
    model=CoherenceNet(enc,device); model.to(device)
    opt=AdamW(model.parameters(),lr=2e-5,eps=1e-8)
    os.makedirs(a.checkpoints_path,exist_ok=True)
    train(model,Subset(full,tr_idx),vdl,opt,a.epochs,a.margin,device,
          a.checkpoints_path,a.batch_size,nw)

if __name__=="__main__":
    main()
'''
ast.parse(TRAIN_PY)
open(os.path.join(REPO_DIR,'train.py'),'w').write(TRAIN_PY)
print('train.py STEP-RESUME 재작성 완료: 그 epoch의 그 step으로 정확 점프, '
      '단일 LR, 재계산 0, eval 호환 cpt_*.pth, 워커, tqdm 10s. 알고리즘 불변.')

In [ ]:
# [3] CM coherence 학습 — 논문 HP, STEP-RESUME (train.py 는 [2e] 가 작성)
#   epochs=10 batch=24 margin=1 encoder=aws-ai/dse-bert-base
#   AdamW(lr2e-5,eps1e-8). transformers 4.39.3(<4.40, AdamW 존재).
#   첫 실행 [fresh] 단일 LR 스케줄 시작 → 멈췄다 재실행 시 [resume]
#   '그 epoch 의 그 step'으로 점프(재계산 0). CKPT_DIR(=[2d] 전용폴더)에
#   cpt_<gstep>.pth(순수 model, eval 호환) + resume.pt(full state) 저장.
#   '[DONE] 10 epoch 완료' = 논문 등가. (seed=42 로 재현·resume 가능)
assert have_dd(), 'DailyDialog 원본 준비 안 됨 — [2]/[2b] 먼저'
EPOCHS, BATCH, MARGIN = 10, 24, 1
ENCODER = 'aws-ai/dse-bert-base'
try:
    CKPT_DIR                       # [2d] Drive 전용폴더
except NameError:
    CKPT_DIR = os.path.join(REPO_DIR, 'checkpoints')   # [2d] 미실행 시 로컬
os.makedirs(CKPT_DIR, exist_ok=True)
print('CKPT_DIR =', CKPT_DIR)
!python train.py -t {DD_DIR}/ -e {ENCODER} -s {CKPT_DIR} -m {MARGIN} -r {EPOCHS} -b {BATCH}

import glob, re
# resume.pt 제외 — cpt_<gstep>.pth 만, gstep 최대(최다학습) 선택
cks = glob.glob(os.path.join(CKPT_DIR, 'cpt_*.pth'))
def _step(p):
    m = re.search(r'cpt_(\d+)\.pth$', os.path.basename(p))
    return int(m.group(1)) if m else -1
cks = sorted([p for p in cks if _step(p) >= 0], key=_step)
print('\ncheckpoints:', [os.path.basename(c) for c in cks[-5:]],
      '(총', len(cks), ')')
assert cks, '체크포인트 미생성 — 위 학습 로그 확인'
BEST_CKPT = cks[-1]                # 최대 gstep = 가장 많이 학습된 순수 model
print('use BEST_CKPT =', BEST_CKPT, '(gstep=%d)' % _step(BEST_CKPT))

In [ ]:
# [4] 평가 — segment.py 로 NSP(zero-shot) + CM(학습 ckpt)
#   재현: CM = 논문 본방법(학습된 DSE-BERT coherence model).
#   NSP baseline 은 논문대로 bert-base-uncased (DSE-BERT 아님).
import glob, re
EVAL_DIR = os.path.join(REPO_DIR, 'data', 'eval')
all_eval = {os.path.basename(p).lower(): p for p in glob.glob(os.path.join(EVAL_DIR, '*.json'))}
wanted = {}
for key, pat in [('dialseg711', 'dialseg'), ('tiage', 'tiage')]:
    hit = next((v for k, v in all_eval.items() if pat in k), None)
    if hit:
        wanted[key] = hit
print('eval targets:', {k: os.path.basename(v) for k, v in wanted.items()})

def run_segment(data_json, encoder, mode):
    """segment.py 실행 후 stdout 에서 Pk/WD/F1 파싱."""
    import subprocess
    out = subprocess.run(
        ['python', 'segment.py', '-t', data_json, '-e', encoder, '-m', mode],
        capture_output=True, text=True, cwd=REPO_DIR)
    txt = out.stdout + '\n' + out.stderr
    def grab(label):
        m = re.search(label + r'[^0-9-]*([0-9]*\.?[0-9]+)', txt, re.I)
        return float(m.group(1)) if m else None
    res = {'pk': grab('P_?k'), 'wd': grab('WindowDiff|Windiff|WD'), 'f1': grab('F1')}
    if all(v is None for v in res.values()):
        print(f'--- segment.py 출력 파싱 실패 ({mode}) ---\n', txt[-1500:])
    return res

rows = []
for ds, path in wanted.items():
    rows.append(('NSP (zero-shot)', ds, run_segment(path, 'bert-base-uncased', 'NSP')))
    rows.append(('CM (trained, 논문 본방법)', ds, run_segment(path, BEST_CKPT, 'CM')))
for label, ds, r in rows:
    print(f'{ds:11s} {label:24s} Pk={r["pk"]} WD={r["wd"]} F1={r["f1"]}')

In [ ]:
# [4b] TextTiling 베이스라인 (nltk) — 같은 eval json 에 동일 Pk/WD/F1 공식
from nltk.tokenize import TextTilingTokenizer
from nltk.metrics import pk as nltk_pk, windowdiff as nltk_wd
from sklearn.metrics import f1_score
import numpy as np, json

def load_eval(path):
    data = json.load(open(path))
    items = data if isinstance(data, list) else data.get('data', data)
    out = []
    for d in (items.values() if isinstance(items, dict) else items):
        utts = d['utterances']
        segs = set(d.get('segments', d.get('segment', [])))
        yt = [1 if i in segs else 0 for i in range(len(utts))]
        if yt: yt[-1] = 0
        if len(utts) >= 2: out.append((utts, yt))
    return out

def official_pk_wd(yt, yp):
    n_seg = sum(yt) + 1
    k = max(2, int(round(len(yt) / n_seg / 2)))
    ts, ps = ''.join(map(str, yt)), ''.join(map(str, yp))
    return float(nltk_pk(ts, ps, k=k)), float(nltk_wd(ts, ps, k=k))

def texttiling_pred(utts):
    tt = TextTilingTokenizer(w=10, k=6)
    try:
        tiles = tt.tokenize('\n\n'.join(u.strip() or '.' for u in utts))
    except Exception:
        return [0] * len(utts)
    pred = []
    for t in tiles:
        lines = [x for x in t.strip().split('\n\n') if x != '']
        if not lines: continue
        pred += [0] * len(lines)
        pred[-1] = 1
    pred = (pred + [0] * len(utts))[:len(utts)]
    if pred: pred[-1] = 0
    return pred

for ds, path in wanted.items():
    dia = load_eval(path)
    pks, wds, g, p = [], [], [], []
    for utts, yt in dia:
        yp = texttiling_pred(utts)
        a, b = official_pk_wd(yt, yp); pks.append(a); wds.append(b)
        g += yt; p += yp
    r = {'pk': float(np.mean(pks)), 'wd': float(np.mean(wds)),
         'f1': float(f1_score(g, p, zero_division=0))}
    rows.append(('TextTiling (nltk)', ds, r))
    print(f'{ds:11s} TextTiling (nltk)  Pk={r["pk"]:.3f} WD={r["wd"]:.3f} F1={r["f1"]:.3f}')

In [ ]:
# [5] 결과 표 + ckpt 묶어서 다운로드
import datetime, shutil, json, torch, transformers
TR_VER = transformers.__version__
REPRO = 'full' if TR_VER == '4.27.4' else f'near (transformers {TR_VER})'
step_n = ''.join(ch for ch in os.path.basename(BEST_CKPT) if ch.isdigit())
lines = ['# CSM 준-완벽재현 — lxing532 CM / NSP / TextTiling\n',
         f'date: {datetime.date.today()} | encoder(CM): {ENCODER} | epochs(target): {EPOCHS} '
         f'| batch: {BATCH} | margin: {MARGIN} | device: {DEVICE}\n',
         f'재현등급: {REPRO}. ckpt=cpt_{step_n} (누적 step={step_n}). '
         f'transformers={TR_VER}(AdamW 무패치,<4.40) / torch={torch.__version__}(Colab 완화). '
         'DailyDialog = 원본 ijcnlp_dailydialog (topic 포함) verbatim.\n',
         '※ step-resume(opt+sched+gstep+epoch 복원, seed=42 pseudo/split/order 결정화) '
         '-> 단일 연속 LR 10ep 등가, 그 epoch/step 정확점프. perf(forward 배치화: '
         'train dropout RNG 상이 = 분포상 동등). '
         '레포 seed 미고정 → 비결정성. metric Pk/WD↓ F1↑ (NSP/CM=segment.py, TextTiling=nltk).\n',
         '| method | dataset | Pk ↓ | WD ↓ | F1 ↑ |',
         '|---|---|---:|---:|---:|']
def fmt(x): return f'{x:.3f}' if isinstance(x, (int, float)) else str(x)
for label, ds, r in rows:
    lines.append(f'| {label} | {ds} | {fmt(r["pk"])} | {fmt(r["wd"])} | {fmt(r["f1"])} |')
results_md = '\n'.join(lines) + '\n'
open('/content/results.md', 'w').write(results_md)
print(results_md)

BUNDLE = '/content/csm_artifacts'
os.makedirs(BUNDLE, exist_ok=True)
shutil.copy(BEST_CKPT, os.path.join(BUNDLE, 'csm_cm_' + os.path.basename(BEST_CKPT)))
shutil.copy('/content/results.md', os.path.join(BUNDLE, 'results.md'))
with open(os.path.join(BUNDLE, 'meta.json'), 'w') as f:
    json.dump({'repo': 'lxing532/Dialogue-Topic-Segmenter',
               'repro_grade': REPRO,
               'transformers': TR_VER, 'torch': torch.__version__,
               'transformers_note': 'paper pin 4.27.4 not installable on Colab py3.12 → 4.39.3 (AdamW present, unpatched)',
               'perf_patch': 'batched CoherenceNet.forward + DataLoader workers (math-identical, no learning change)',
               'training_mode': 'TRUE step-resume: resume.pt(model+opt+sched+gstep+epoch+sie); single continuous LR over full 10ep; seed=42 (split+epoch-order+data_utils pseudo) -> exact epoch/step jump, no recompute ~= single continuous 10-epoch run.',
               'cumulative_step': step_n,
               'encoder_CM': ENCODER, 'epochs_target': EPOCHS, 'batch': BATCH,
               'margin': MARGIN, 'optimizer': 'transformers.AdamW lr2e-5 eps1e-8 (unpatched)',
               'dailydialog': 'original ijcnlp_dailydialog (verbatim, topic incl.)',
               'seed': 'NOT fixed by repo (run-to-run variance)',
               'ckpt': os.path.basename(BEST_CKPT)}, f, indent=2)
zip_path = shutil.make_archive('/content/csm_artifacts', 'zip', BUNDLE)
print('bundle:', zip_path, '(', round(os.path.getsize(zip_path)/1e6, 1), 'MB )')
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('자동 다운로드 불가 — 좌측 파일창에서', zip_path, '수동 다운로드:', e)

## 가져온 뒤 (로컬 Hi-EM)

`csm_artifacts.zip` 안 `csm_cm_*.pth` 가 학습된 CM 체크포인트.

1. `csm_cm_*.pth` 를 로컬 `outputs/runs/_misc/` (또는 지정 경로)에 둠
2. 로컬에서 이 ckpt 로 **superseg + tiage + dialseg711** 을 Hi-EM 공식 SuperDialseg Pk/WD/F1 harness 로 재평가 → 기존 `outputs/experiments/.../REPORT.md` 와 동일 metric 으로 통합 (Claude 가 처리)
3. `results.md` 는 Colab 단계 sanity 표 (metric provenance 섞임 + topic 합성 → 최종 비교는 로컬 공식 수치 기준, REPORT 에 topic 합성 한계 명시)